# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from student.data import validate_splits, read_split
from student.visualize import annotate

print(validate_splits("data"))
train_rows = read_split("data/splits/train.jsonl")
# Ground-truth preview overlays can be saved after inspecting train_rows.


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from student.labels import yolo_to_box, objects_to_answer, box_to_pixels

print(objects_to_answer([{"label": "pothole", "yolo": [0.5, 0.5, 0.2, 0.2]}]))


## 3. E0: zero-shot

In [ ]:
from student.model import load_model
from student.infer import generate_split, read_split

model, processor = load_model()
test_rows = read_split("data/splits/test.jsonl")
generate_split(model, processor, test_rows, out_path="work/e0.jsonl")


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from student.data import encode_example, collate, PROMPT
# encode_example(processor, image, objects); collate(examples, processor.tokenizer.pad_token_id)


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from student.model import apply_setup, audit_model
# model = apply_setup(model, "e1")  # use a fresh pretrained model for each experiment


## 6. Vòng huấn luyện

In [ ]:
from student.train import train, save_checkpoint, restore_extra_trainables
# Train from fresh model on train split only; record returned resource measurements.


## 7. Suy luận và chấm E1–E3

In [ ]:
from student.infer import generate_split, read_split
# Restore each checkpoint in a fresh base model before generating its predictions.


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: follow configs/e4.yaml and change one factor only. E5 is optional.
